In [36]:
# ==========================================================
# Notebook 11: End-to-End AI ATS System
# ==========================================================

import os
import re
import faiss
import pickle
import pdfplumber
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer, CrossEncoder

from sklearn.metrics.pairwise import cosine_similarity

In [37]:
# Load Sentence-BERT
sbert_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Load Cross-Encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Load FAISS index
faiss_index = faiss.read_index("indexes/faiss_resume_index.bin")

# Load metadata
with open("indexes/resume_metadata.pkl", "rb") as file:

    metadata = pickle.load(file)

print("System Loaded Successfully.")

System Loaded Successfully.


In [38]:
def parse_resume(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text

In [39]:
def extract_sections(resume_text):

    sections = {"skills": "", "experience": "", "education": ""}

    skills_match = re.search(
        r"skills(.*?)(experience|education|$)",
        resume_text,
        flags=re.IGNORECASE | re.DOTALL,
    )

    experience_match = re.search(
        r"experience(.*?)(education|$)", resume_text, flags=re.IGNORECASE | re.DOTALL
    )

    education_match = re.search(
        r"education(.*)$", resume_text, flags=re.IGNORECASE | re.DOTALL
    )

    if skills_match:
        sections["skills"] = skills_match.group(1)

    if experience_match:
        sections["experience"] = experience_match.group(1)

    if education_match:
        sections["education"] = education_match.group(1)

    return sections

In [40]:
def anonymize_resume(text):

    # Remove emails
    text = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", "[EMAIL]", text)

    # Remove phone numbers
    text = re.sub(r"(\+?\d[\d\s-]{8,}\d)", "[PHONE]", text)

    # Mask candidate name (first line)
    lines = text.split("\n")

    if len(lines) > 0:
        lines[0] = "[CANDIDATE_NAME]"

    text = "\n".join(lines)

    return text

In [41]:
def create_resume_embedding(sections):

    combined_text = (
        sections["skills"] + " " + sections["experience"] + " " + sections["education"]
    )

    embedding = sbert_model.encode(combined_text)

    return embedding

In [42]:
def retrieve_candidates(job_description, top_k=10):

    # Create Job Description embedding
    jd_embedding = sbert_model.encode(job_description)

    jd_embedding = np.array([jd_embedding]).astype("float32")

    faiss.normalize_L2(jd_embedding)

    # Search FAISS
    scores, indices = faiss_index.search(
        jd_embedding,
        top_k,
    )

    candidates = []

    for score, idx in zip(
        scores[0],
        indices[0],
    ):

        candidate = {
            "candidate": metadata["file_names"][idx],
            "faiss_score": float(score),
        }

        # Only add resume text if available
        if "resume_text" in metadata:
            candidate["resume_text"] = metadata["resume_text"][idx]

        candidates.append(candidate)

    return candidates

In [43]:
def rerank_candidates(
    job_description,
    candidates,
):

    # If metadata has no resume text,
    # skip reranking.
    if len(candidates) == 0:
        return candidates

    if "resume_text" not in candidates[0]:

        for candidate in candidates:
            candidate["cross_score"] = candidate["faiss_score"]

        return candidates

    pairs = [
        (
            job_description,
            candidate["resume_text"],
        )
        for candidate in candidates
    ]

    scores = cross_encoder.predict(pairs)

    for i in range(len(candidates)):
        candidates[i]["cross_score"] = float(scores[i])

    candidates = sorted(
        candidates,
        key=lambda x: x["cross_score"],
        reverse=True,
    )

    return candidates

In [44]:
SKILL_VOCABULARY = [
    "python",
    "sql",
    "machine learning",
    "deep learning",
    "pytorch",
    "langchain",
    "rag",
    "docker",
    "kubernetes",
    "nlp",
]

In [45]:
def extract_skills(text):

    text = text.lower()

    skills = []

    for skill in SKILL_VOCABULARY:

        if skill in text:

            skills.append(skill)

    return skills

In [46]:
def skill_gap_analysis(job_description, candidate_text):

    jd_skills = extract_skills(job_description)

    candidate_skills = extract_skills(candidate_text)

    matched = list(set(jd_skills) & set(candidate_skills))

    missing = list(set(jd_skills) - set(candidate_skills))

    return {
        "matched_skills": matched,
        "missing_skills": missing,
        "coverage": round(100 * len(matched) / max(1, len(jd_skills)), 2),
    }

In [47]:
def ai_resume_matcher(
    job_description,
    top_k=5,
):

    # Step 1: FAISS Retrieval
    candidates = retrieve_candidates(
        job_description,
        top_k,
    )

    # Step 2: Cross Encoder (if possible)
    candidates = rerank_candidates(
        job_description,
        candidates,
    )

    # Step 3: Skill Gap Analysis
    final_results = []

    for candidate in candidates:

        if "resume_text" in candidate:

            gap_report = skill_gap_analysis(
                job_description,
                candidate["resume_text"],
            )

            candidate.update(gap_report)

        else:
            candidate["matched_skills"] = []

            candidate["missing_skills"] = []

            candidate["coverage"] = 0.0

        final_results.append(candidate)

    return final_results

In [48]:
job_description = """
Need a Machine Learning Engineer with:

- Python
- NLP
- LangChain
- RAG
- Docker
"""

results = ai_resume_matcher(
    job_description,
    top_k=5,
)

results

[{'candidate': '25547145.pdf',
  'faiss_score': 0.34727129340171814,
  'cross_score': 0.34727129340171814,
  'matched_skills': [],
  'missing_skills': [],
  'coverage': 0.0},
 {'candidate': '33527446.pdf',
  'faiss_score': 0.3283308446407318,
  'cross_score': 0.3283308446407318,
  'matched_skills': [],
  'missing_skills': [],
  'coverage': 0.0},
 {'candidate': '78257294.pdf',
  'faiss_score': 0.3254126310348511,
  'cross_score': 0.3254126310348511,
  'matched_skills': [],
  'missing_skills': [],
  'coverage': 0.0},
 {'candidate': '24294778.pdf',
  'faiss_score': 0.32309573888778687,
  'cross_score': 0.32309573888778687,
  'matched_skills': [],
  'missing_skills': [],
  'coverage': 0.0},
 {'candidate': '98559931.pdf',
  'faiss_score': 0.3222368359565735,
  'cross_score': 0.3222368359565735,
  'matched_skills': [],
  'missing_skills': [],
  'coverage': 0.0}]

In [49]:
leaderboard = pd.DataFrame(results)

leaderboard[["candidate", "faiss_score", "cross_score", "coverage"]]

,candidate,faiss_score,cross_score,coverage
0,25547145.pdf,0.347271,0.347271,0.0
1,33527446.pdf,0.328331,0.328331,0.0
2,78257294.pdf,0.325413,0.325413,0.0
3,24294778.pdf,0.323096,0.323096,0.0
4,98559931.pdf,0.322237,0.322237,0.0
